# Ridge Regression: Disease Progression with Correlated Predictors

**Family:** Supervised learning | **Level:** Intermediate | **Dataset:** `diabetes` | **Runs on:** CPU, about 1 minute(s)

Shrink coefficients with an L2 penalty and watch the bias variance trade-off on diabetes data.

## Table of Contents

1. [Learning Objectives](#1.-Learning-Objectives)
2. [Real-World Problem](#2.-Real-World-Problem)
3. [Research Paper](#3.-Research-Paper)
4. [Mathematical Foundation](#4.-Mathematical-Foundation)
5. [Dataset Introduction](#5.-Dataset-Introduction)
6. [Exploratory Data Analysis](#6.-Exploratory-Data-Analysis)
7. [Data Preprocessing](#7.-Data-Preprocessing)
8. [Algorithm Implementation](#8.-Algorithm-Implementation)
9. [Model Training](#9.-Model-Training)
10. [Evaluation](#10.-Evaluation)
11. [Visualization](#11.-Visualization)
12. [Interpretation](#12.-Interpretation)
13. [Experiments and Challenges](#13.-Experiments-and-Challenges)
14. [Research Discussion](#14.-Research-Discussion)
15. [Summary and Further Reading](#15.-Summary-and-Further-Reading)

## 1. Learning Objectives

**After this notebook you will understand:**

- What the L2 penalty does to coefficients
- Why scaling matters before penalised regression
- How to read a coefficient path

**Prerequisites:**

- Linear regression notebook
- Train, validation and test splits

**Skills you will have afterwards:**

- Choose a hyperparameter on validation data
- Compare with a baseline
- Run a sensitivity experiment

## 2. Real-World Problem

**Context.** Clinical studies measure many correlated blood markers per patient.

**The problem.** Predict a quantitative measure of diabetes progression one year after baseline from ten baseline features.

**Why it matters.** Stable and accurate risk scores help target follow up, and unstable coefficients make a score untrustworthy.

**Why machine learning.** Markers interact and no closed form biological model is available.

**Why this algorithm.** The serum measures are strongly correlated, which is exactly where ridge stabilises least squares.

## 3. Research Paper

Entries are grouped by type. Bibliographic details marked *verified* were checked against the publisher record (Crossref) or the arXiv abstract page on 2026-11-17. Entries marked *not verified* come from the author's knowledge and must be checked before citing.

### Ridge Regression: Biased Estimation for Nonorthogonal Problems

- **Type:** foundational paper
- **Authors:** Hoerl, A. E. and Kennard, R. W.
- **Year and venue:** 1970, Technometrics 12(1):55-67
- **DOI or URL:** https://doi.org/10.1080/00401706.1970.10488634
- **Verification:** Crossref 2026-11-17
- **Research objective:** Stabilise least squares when predictors are nearly collinear.
- **Methodology:** Add a multiple of the identity to X'X before inversion.
- **Dataset:** Illustrative examples.
- **Main findings:** A small bias can cut the variance and the mean squared error of coefficients.
- **Limitations:** Keeps every predictor, so it does not select variables.

### Least angle regression

- **Type:** foundational paper
- **Authors:** Efron, B., Hastie, T., Johnstone, I. and Tibshirani, R.
- **Year and venue:** 2004, The Annals of Statistics 32(2):407-499
- **DOI or URL:** https://doi.org/10.1214/009053604000000067
- **Verification:** Crossref 2026-11-17
- **Research objective:** Give an efficient path algorithm for the lasso and related methods.
- **Methodology:** Move coefficients along the direction equiangular to the active predictors.
- **Dataset:** The diabetes progression data of 442 patients used in this repository.
- **Main findings:** The full lasso path costs about the same as one least squares fit.
- **Limitations:** Sensitive to noise when predictors are strongly correlated.

**Relationship to this notebook.** Hoerl and Kennard introduce ridge for nearly collinear predictors. Efron et al. use the same diabetes data (442 patients) to introduce least angle regression, so the data matches the literature in structure, though not the exact protocol. This notebook does not reproduce the experimental protocol of any listed paper. It uses a different dataset and split, so numbers are not comparable with the paper.

## 4. Mathematical Foundation

**Objective.**

$$\hat\beta^{\text{ridge}} = \arg\min_\beta \; \lVert y - A\beta \rVert_2^2 + \lambda \lVert \beta \rVert_2^2, \qquad \lambda \ge 0$$

The intercept is not penalised.

**Closed form.** $\hat\beta = (X^\top X + \lambda I)^{-1} X^\top y$. Adding $\lambda I$ makes the matrix invertible even when $X^\top X$ is nearly singular.

**Effect.** In the SVD basis, each component is shrunk by $d_j^2 / (d_j^2 + \lambda)$, so directions of low variance are shrunk most.

**Assumptions.** Same as least squares, plus standardised features so that the penalty treats them equally.

**Limitations.** Never sets a coefficient exactly to zero, so it does not select features. $\lambda$ must be tuned.

## 5. Dataset Introduction

**Diabetes progression** from Efron, Hastie, Johnstone and Tibshirani (2004), shipped with scikit-learn. 442 patients, 10 baseline features (`age`, `sex`, `bmi`, `bp`, and six blood serum measures `s1` to `s6`), target is a quantitative measure of disease progression one year after baseline. The provider centred and scaled the features. Data terms of the original source are not stated by the provider. Limits: small sample, no time dimension, `sex` is a standardised number.

The table printed below the next cell comes from the dataset registry in `src/ml_lab/data/registry.py`. Licence terms belong to the data provider, not to this repository.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from sklearn.base import clone
from sklearn.dummy import DummyClassifier, DummyRegressor
from sklearn.inspection import permutation_importance
from sklearn.model_selection import cross_validate
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from ml_lab.data import load_dataset, missing_report
from ml_lab.preprocessing import split_three_way, build_preprocessor, make_model_pipeline, feature_names
from ml_lab.evaluation import evaluate_estimator, compare_models
from ml_lab.experiments import ExperimentRecord, save_figure
from ml_lab.utils import set_seed, print_environment
from ml_lab import visualization as viz

NOTEBOOK_ID = "ridge_regression"
SEED = 42
set_seed(SEED)
viz.apply_style()
print_environment()

In [ ]:
bundle = load_dataset("diabetes")
data = bundle.frame
TASK = bundle.task
TARGET = bundle.target
print("task:", TASK, "| target:", TARGET, "| shape:", data.shape)
display(pd.Series(bundle.describe_source(), name="source").to_frame())
display(data.head())

## 6. Exploratory Data Analysis

We look at size, types, missing values, the target, feature distributions, correlation and outliers before any modelling. 

In [ ]:
print("Rows and columns:", data.shape)
display(data.dtypes.value_counts().rename("columns by dtype").to_frame())
display(data.describe().T.round(3).head(20))
missing = missing_report(data)
print("Columns with missing values:", len(missing))
display(missing)
print("Duplicate rows:", int(data.duplicated().sum()))

In [ ]:
fig = viz.plot_target(data[TARGET], TASK)
plt.show()
fig = viz.plot_distributions(data.drop(columns=[TARGET]))
plt.show()
fig = viz.plot_correlation(data)
plt.show()
display(viz.outlier_table(data.drop(columns=[TARGET])).head(8))

## 7. Data Preprocessing

Rules used here:

- Split first, then fit imputers, scalers and encoders on the training rows only. This prevents data leakage.
- The test set is touched once, at the end.
- Order inside the pipeline: impute, scale or encode, then model.


In [ ]:

X, y = data.drop(columns=[TARGET]), data[TARGET]
X_train, X_val, X_test, y_train, y_val, y_test = split_three_way(X, y, task=TASK, seed=SEED)
print("train / validation / test rows:", len(X_train), len(X_val), len(X_test))
if TASK == "classification":
    split_rates = pd.DataFrame({"train": y_train.value_counts(normalize=True),
                                "validation": y_val.value_counts(normalize=True),
                                "test": y_test.value_counts(normalize=True)}).round(3)
    display(split_rates)
prep = build_preprocessor(X_train, scale=True, encode="onehot")
Xtr = prep.fit_transform(X_train)   # fit on train only
Xva, Xte = prep.transform(X_val), prep.transform(X_test)
print("transformed train shape:", Xtr.shape)

## 8. Algorithm Implementation

### 8.1 Concept

Ridge trades a little bias for a large drop in variance. As $\lambda$ grows all coefficients move smoothly toward zero.

### 8.2 Python implementation from scratch

The closed form is short, so we implement it and compare with scikit-learn.

In [ ]:
ytr = y_train.values

def ridge_fit(X, y, alpha):
    A = np.c_[np.ones(len(X)), X]
    penalty = alpha * np.eye(A.shape[1])
    penalty[0, 0] = 0                      # keep intercept free
    return np.linalg.solve(A.T @ A + penalty, A.T @ y)

mine = ridge_fit(Xtr, ytr, alpha=10.0)
reference = Ridge(alpha=10.0).fit(Xtr, ytr)
print("max coefficient gap vs sklearn:", np.abs(mine[1:] - reference.coef_).max())

### 8.3 Library implementation

The estimator is wrapped in a pipeline so imputation, scaling and encoding are fitted on training rows only.

### 8.4 Important parameters

| Parameter | Meaning |
|---|---|
| `alpha` | Penalty strength $\lambda$. Larger means more shrinkage. |
| `solver` | Numerical method. `auto` is fine here. |

## 9. Model Training

Procedure: fit on the training rows, choose one hyperparameter on the validation rows, refit on training plus validation, then evaluate once on the test rows. The random seed is fixed (`SEED = 42`). 

In [ ]:
SELECT = "RMSE"
LOWER_IS_BETTER = True
SCORING = "neg_root_mean_squared_error"
SWEEP_PARAM = "model__alpha"
SWEEP_VALUES = [0.001, 0.01, 0.1, 1, 3, 10, 30, 100]

def build_pipeline():
    return make_model_pipeline(X_train, Ridge(alpha=1.0), scale=True, encode="onehot")

pipe = build_pipeline()
print(pipe.named_steps["model"])
rows = []
for value in SWEEP_VALUES:
    candidate = clone(pipe).set_params(**{SWEEP_PARAM: value}).fit(X_train, y_train)
    rows.append({SWEEP_PARAM: value, **evaluate_estimator(candidate, X_val, y_val, TASK)})
sweep = pd.DataFrame(rows).set_index(SWEEP_PARAM)
display(sweep[[c for c in [SELECT, "R2", "MAE", "Accuracy", "F1", "ROC_AUC", "PR_AUC"] if c in sweep.columns]].round(4))
best_value = sweep[SELECT].idxmin() if LOWER_IS_BETTER else sweep[SELECT].idxmax()
print("best", SWEEP_PARAM, "on validation:", best_value)
final_model = clone(pipe).set_params(**{SWEEP_PARAM: best_value})
X_trainval, y_trainval = pd.concat([X_train, X_val]), pd.concat([y_train, y_val])
final_model.fit(X_trainval, y_trainval)   # refit before test
print("refit on train plus validation rows")

## 10. Evaluation

Regression metrics: MAE (average absolute error in target units), RMSE (square root of the mean squared error, same unit, punishes large errors), R-squared (share of variance explained, can be negative on new data) and MAPE when the target is never zero.

A dummy model gives the floor every real model must beat.

In [ ]:
baseline = DummyRegressor(strategy="mean") if TASK == "regression" else DummyClassifier(strategy="prior")
baseline = make_model_pipeline(X_train, baseline, scale=False).fit(X_trainval, y_trainval)
test_metrics = evaluate_estimator(final_model, X_test, y_test, TASK)
baseline_metrics = evaluate_estimator(baseline, X_test, y_test, TASK)
print("Test set, scored once after model choice")
display(compare_models({"baseline": baseline_metrics, "Ridge": test_metrics}))
cv = cross_validate(final_model, X_trainval, y_trainval, cv=5, scoring=SCORING, n_jobs=1)
cv_scores = cv["test_score"]
print(f"5-fold CV {SCORING}: mean {cv_scores.mean():.4f}, std {cv_scores.std(ddof=1):.4f}")
record = ExperimentRecord(notebook=NOTEBOOK_ID, family="supervised", algorithm="Ridge", dataset="diabetes",
                          task=TASK, seed=SEED, config={"param": SWEEP_PARAM, "value": str(best_value)},
                          metrics={k: float(v) for k, v in test_metrics.items()},
                          extra={"baseline": {k: float(v) for k, v in baseline_metrics.items()},
                                 "cv_mean": float(cv_scores.mean()), "cv_std": float(cv_scores.std(ddof=1)),
                                 "cv_scoring": SCORING, "select_metric": SELECT})
print("saved", record.save().name)

## 11. Visualization

All plots use the test rows unless the title says otherwise.

In [ ]:
pred = final_model.predict(X_test)
fig = viz.plot_pred_vs_actual(y_test, pred, unit="(progression score)")
save_figure(fig, NOTEBOOK_ID, "pred_vs_actual"); plt.show()
fig = viz.plot_residuals(y_test, pred, unit="(progression score)")
save_figure(fig, NOTEBOOK_ID, "residuals"); plt.show()
fig = viz.plot_learning_curve(final_model, X_trainval, y_trainval, scoring=SCORING)
save_figure(fig, NOTEBOOK_ID, "learning_curve"); plt.show()

## 12. Interpretation

Permutation importance shuffles one feature and measures how much the error grows. Correlated features share importance, so read it as a ranking of groups. Check the residual plot for curvature or funnel shapes, which mean the model misses structure or the error variance changes with the prediction.

In [ ]:
perm = permutation_importance(final_model, X_test, y_test, scoring=SCORING, n_repeats=10, random_state=SEED)
importance = pd.Series(perm.importances_mean, index=X_test.columns).sort_values(ascending=False)
fig = viz.plot_importance(importance.index, importance.values, title="Permutation importance on the test set",
                          top=12, xlabel="Drop in " + SCORING + " when shuffled")
save_figure(fig, NOTEBOOK_ID, "importance"); plt.show()
print("Top features:", ", ".join(importance.index[:5]))
errors = (y_test.values - final_model.predict(X_test)) if TASK == "regression" else (y_test.values != final_model.predict(X_test))
if TASK == "regression":
    worst = pd.DataFrame({"actual": y_test.values, "predicted": final_model.predict(X_test)}, index=y_test.index)
    worst["abs_error"] = (worst["actual"] - worst["predicted"]).abs()
    print("Largest test errors:")
    display(worst.sort_values("abs_error", ascending=False).head(5).round(3))
else:
    print("Misclassified test rows:", int(errors.sum()), "of", len(errors))
print()
print(f"Test {SELECT}: {test_metrics[SELECT]:.4f} | baseline {SELECT}: {baseline_metrics[SELECT]:.4f}")
print(f"Cross validation spread (std): {cv_scores.std(ddof=1):.4f}. A test gap larger than this spread is meaningful.")

## 13. Experiments and Challenges

### Experiment 1: Coefficient path

Plot every coefficient against $\lambda$ on a log axis.

In [ ]:
alphas = np.logspace(-3, 3, 40)
paths = np.array([Ridge(alpha=a).fit(Xtr, y_train).coef_ for a in alphas])
names = feature_names(clone(pipe).fit(X_train, y_train))
plt.plot(alphas, paths)
plt.xscale("log"); plt.xlabel("alpha (penalty strength)"); plt.ylabel("Coefficient (progression units per std)")
plt.legend([n.split("__")[-1] for n in names], ncol=2, fontsize=7); plt.title("Ridge coefficient paths"); plt.show()

### Experiment 2: Ridge versus ordinary least squares under added collinearity

Add a noisy copy of `bmi` and compare cross validated RMSE of OLS and ridge.

In [ ]:
from sklearn.model_selection import cross_val_score
def cv_rmse(model, X_data=None, y_data=None):
    scores = cross_val_score(model, X_data if X_data is not None else X_trainval,
                             y_data if y_data is not None else y_trainval,
                             cv=5, scoring="neg_root_mean_squared_error")
    return -scores.mean(), scores.std(ddof=1)
from sklearn.linear_model import LinearRegression
rng = np.random.default_rng(SEED)
X_dup = X_trainval.copy()
X_dup["bmi_copy"] = X_dup["bmi"] + rng.normal(0, 0.002, len(X_dup))
for name, model in [("OLS", LinearRegression()), ("Ridge alpha=10", Ridge(alpha=10))]:
    mean, std = cv_rmse(make_model_pipeline(X_dup, model), X_dup)
    print(f"{name}: CV RMSE {mean:.2f} +- {std:.2f}")

### Experiment 3: Why scaling matters

Fit ridge with and without scaling on a version of the data where one feature is multiplied by 1000. The penalty is not scale invariant.

In [ ]:
from sklearn.model_selection import cross_val_score
def cv_rmse(model, X_data=None, y_data=None):
    scores = cross_val_score(model, X_data if X_data is not None else X_trainval,
                             y_data if y_data is not None else y_trainval,
                             cv=5, scoring="neg_root_mean_squared_error")
    return -scores.mean(), scores.std(ddof=1)
X_big = X_trainval.copy()
X_big["bmi"] = X_big["bmi"] * 1000
for scaled in [True, False]:
    mean, std = cv_rmse(make_model_pipeline(X_big, Ridge(alpha=1.0), scale=scaled), X_big)
    print(f"scaled={scaled}: CV RMSE {mean:.2f} +- {std:.2f}")

### Exercises for learners

1. Repeat the notebook with `SEED = 7` and report how much the headline metric changes.
2. Add one more feature or remove one and explain the change using cross validation.
3. Replace the validation sweep with `GridSearchCV` and compare the chosen value.

## 14. Research Discussion

Hoerl and Kennard motivate ridge for near collinear predictors, and our coefficient path shows the smooth shrinkage they describe. Efron et al. analyse lasso type paths on the same patients, which the lasso notebook picks up.

**Dataset and method differences.** The papers above use other datasets, other splits and often other implementations. We do not claim that our numbers agree or disagree with them. What this notebook can show is whether the behaviour described in the paper (for example the effect of a parameter) appears on our data.

**Possible improvements.** Select alpha with `RidgeCV` and nested cross validation, and report intervals with a bootstrap.

**Research limitations.** One dataset, one split seed for the headline number, and small test sets for the UCI data. Cross validation spread is reported to show how unstable a single split can be.

## 15. Summary and Further Reading

**Key takeaways**

- Ridge shrinks but does not select.
- Scale features first.
- Tune alpha on validation or cross validation data.

**Use cases.** Many correlated predictors, small samples, stable scoring models.

**Limitations.** No feature selection, linear only.

**Recommended next notebook.** Lasso Regression.

**Further reading.** See `references.md` in this folder and the library documentation for the classes used above.